In [ ]:
# minimal_real_bench_toggle.py
import numpy as np, scanpy as sc
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.isotonic import IsotonicRegression
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
from scipy.stats import spearmanr, kendalltau


def load_dentategyrus_with_time():
    import scvelo as scv
    adata = scv.datasets.dentategyrus()  # has obs['age(days)'] with P12/P35
    t = adata.obs["age(days)"].to_numpy().astype(float)
    return adata, t


def load_moignard15_with_time():
    adata = sc.datasets.moignard15()
    stages = adata.obs["exp_groups"].astype(str).str.lower().values
    ORDER = [("ps","primitive"), ("np","neural"), ("hf","head"), ("4","4"), ("8","8")]
    def stage_to_num(s):
        for i,(k1,k2) in enumerate(ORDER):
            if k1 in s or k2 in s: return i
        return np.nan
    t = np.array([stage_to_num(s) for s in stages], dtype=float)
    if np.isnan(t).any():
        cats = list(dict.fromkeys(stages))
        t = np.array([cats.index(s) for s in stages], dtype=float)
    return adata, t

def make_features(adata, mode="pca", n_pcs=20, seed=0):
    X = adata.X
    X = X.A if hasattr(X, "A") else X  # dense
    # qPCR: impute per gene, standardize; no extra log
    X = SimpleImputer(strategy="median").fit_transform(X)
    X = StandardScaler(with_mean=True, with_std=True).fit_transform(X)
    if mode == "raw":
        return X
    elif mode == "pca":
        return PCA(n_components=n_pcs, random_state=seed).fit_transform(X)
    else:
        raise ValueError("mode must be 'raw' or 'pca'")

def geodesic_2d(Y, k=15, root_idx=0):
    nn = NearestNeighbors(n_neighbors=k).fit(Y)
    d, I = nn.kneighbors(Y)
    rows = np.repeat(np.arange(Y.shape[0])[:,None], k, axis=1).ravel()
    G = csr_matrix((d.ravel(), (rows, I.ravel())), shape=(Y.shape[0], Y.shape[0]))
    G = G.minimum(G.T)
    return dijkstra(G, directed=False, indices=root_idx)

def benchmark(adata, t, dr, mode="pca", k=15, n_pcs=20, seed=0):
    Z = make_features(adata, mode=mode, n_pcs=n_pcs, seed=seed)
    Y = dr.fit_transform(Z)
    if Y.shape[1] > 2: Y = Y[:, :2]
    root = int(np.nanargmin(t))
    g = geodesic_2d(Y, k=k, root_idx=root)

    spearman = float(spearmanr(g, t, nan_policy="omit").correlation)
    kendall  = float(kendalltau(g, t, nan_policy="omit").correlation)
    #iso = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(g, t)
    #iso_r2 = float(1 - np.var(t - iso.predict(g)) / (np.var(t) + 1e-8))

    #print(f"[mode={mode}] Spearman: {spearman:.3f} | Kendall: {kendall:.3f} | IsotonicR2: {iso_r2:.3f}")
    return {"spearman": spearman, "kendall": kendall}

adata, t = load_dentategyrus_with_time()
X = adata.X
X = X.A if hasattr(X, "A") else X  # dense
X = X.toarray()
X = X.astype(np.float32)   
adata.X = X

for random_state in range(1):

    import umap
    dr = umap.UMAP(n_components=2, random_state=random_state)
    print(benchmark(adata, t, dr, mode="raw"))   # imputed+standardized genes directly

    dr = PCA(n_components=2, random_state=random_state)  # fallback
    print(benchmark(adata, t, dr, mode="raw"))   # imputed+standardized genes directly



    X = adata.X
    # qPCR: impute per gene, standardize; no extra log
    X = SimpleImputer(strategy="median").fit_transform(X)
    X = StandardScaler(with_mean=True, with_std=True).fit_transform(X)


    from sklearn.cluster import KMeans

    clusters = []
    n_clusters_list = [4, 8, 16, 32]
    umap = umap.UMAP(n_components=10, random_state=random_state)
    X_umap = umap.fit_transform(X)
    for n_clusters in n_clusters_list:
        kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init="auto") 
        cluster_labels = kmeans.fit_predict(X_umap)
        clusters.append(cluster_labels)

    from pcc import PCC
    dr = PCC(n_components=2, num_epochs=1000, num_points=1000, pearson=True, spearman=False, beta=20, k_epoch=50, temperature=20, batch_size=4096*2*2, sampling="random", linear_cluster_weight=1.0, cluster=True, random_state=random_state)
    dr.fit_transform = lambda x: PCC.fit_transform(dr, x, clusters)
    print(benchmark(adata, t, dr, mode="raw"))   # imputed+standardized genes directly

    from pcc import PCUMAP
    dr = PCUMAP(device="cuda", random_state=random_state)
    print(benchmark(adata, t, dr, mode="raw"))   # imputed+standardized genes directly





    print("**")


{'spearman': 0.11503634916827948, 'kendall': 0.09394287843988339}
{'spearman': -0.18168870625704528, 'kendall': -0.1483735379574631}


100%|██████████| 1000/1000 [00:01<00:00, 798.86it/s]


{'spearman': -0.18786676587304518, 'kendall': -0.1534259302103253}
{'spearman': -0.2712140692659616, 'kendall': -0.22148314634198307}
**


In [20]:
del dr
import torch
torch.cuda.empty_cache()

In [22]:
X.shape

(3934, 42)